In [ ]:
!git clone https://github.com/Nippani-Meghana/tssl-bp-perturbation

Cloning into 'tssl-bp-perturbation'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


# Libraries


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as f
from time import time
import matplotlib.pyplot as plt

# TSSL-BP Implementation

In [20]:
#Zhang et al. code from github repo TSSL-BP (stonezwr/TSSL-BP)
#Reference: Wenrui Zhang and Peng Li. 2020.
#Temporal spike sequence learning via backpropagation for deep spiking neural networks.
#In Proceedings of the 34th International Conference
#on Neural Information Processing Systems (NIPS '20). Curran Associates Inc.,
#Red Hook, NY, USA, Article 1008, 12022–12033.
class TSSLBPModel(torch.autograd.Function):
  @staticmethod
  def forward(ctx, inputs, parameters):
    #inputs has 5D data inside it
    #inputs last index data is time, which is then stored in n_steps
    shape = inputs.shape
    n_steps = shape[4]

    #u_i[t] = (1 - 1/tau_m)u_i[t-1] + summ_j(wij*a_j[t]) + eta_i[t] ... (1)
    #theta_m = 1/tau_m
    theta_m = 1/parameters['tau_m']
    tau_s = parameters['tau_s']

    theta_s = 1/tau_s
    threshold = parameters['threshold']

    mem = torch.zeros(shape[0], shape[1], shape[2], shape[3])
    syn = torch.zeros(shape[0], shape[1], shape[2], shape[3])

    syns_posts = []
    mems = []
    mem_updates = []
    outputs = []

    for t in range(n_steps):
        #minus u_i[t-1] from eqn(1)
        #we get delta_u_i = -theta_m * u_i[t-1] + summ_j(wij*a_j[t]) + eta_i[t]
        #Need to confirm if this means, that all the inputs are considered except t?
        #Ans: No, inputs[...,t] means only the current time step.
        mem_update = (-theta_m) * mem + inputs[..., t] #where is the reset(eta_i[t])?
        #u_i[t-1] += delta_u_i
        mem += mem_update

        out = mem > threshold
        #Returns True if spike happened else False
        out = out.type(torch.float32)
        #converts boolean to 1 if True else 0

        mems.append(mem.clone())
        #if fired:
        #    mem = 0
        #else:
        #   mem stays
        mem = mem * (1 - out) #This is the reset (eta_i[t])
        outputs.append(out)
        mem_updates.append(mem_update)

        #PSCs --can't find the exact equation for this.
        #I know this is the PSCs that gets sent to the next layer.
        syn = syn + (out - syn) * theta_s #what is this supposed to mean? ...(2)
        #Ans: This is the spike response kernel, in discretized form.
        #tau_s · (da_j(t)/dt) = -a_j(t) + s_j(t)
        #da/dt = (s - a) / tau_s
        #a[t+1] - a[t] = (s[t] - a[t]) · (1/tau_s)
        #a[t+1] = a[t] + (s[t] - a[t]) · theta_s, where theta_s = 1/tau_s
        syns_posts.append(syn)


    mems = torch.stack(mems, dim = 4)
    mem_updates = torch.stack(mem_updates, dim = 4)
    outputs = torch.stack(outputs, dim = 4)
    syns_posts = torch.stack(syns_posts, dim = 4)
    ctx.save_for_backward(mem_updates, outputs, mems, syns_posts, torch.tensor([threshold, tau_s, theta_m]))

    return syns_posts
    #Pending Questions:
        #1. Where is the synaptic model (spike response kernel)?
        #Ans: eq(2)
        #2. tau_s*a_j(t)/dt = - a_j(t) + s_j(t). This equation is continuous.
        #Where is the continuous part?
        #Ans: It gets discretized
        #3. Does this function get called layer by layer, because it returns PSCs.
        #Becasue according to the paper, the PSCs get sent to the next layer?
        #Ans: Will have to look at other files to confirm
    #Summary:
        #1. Input -> LIF Update -> Returns PSCs -> Unweighted PSCs to Next Layer
        #2. Entire code is discrete form




